# Zyro Dynamics HR Help Desk — RAG Challenge
### NxtWave Masterclass | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Deploy a Streamlit chatbot
- Generate your `submission.csv`

## Submission Requirements

1. `submission.csv` — upload on Kaggle
2. Streamlit App URL
3. LangSmith Trace URL

---

> Follow the notebook cells sequentially and complete the sections marked for implementation.

## Cell 1 — Install Dependencies

> ⚠️ Run this cell first before anything else.

This cell installs all required libraries for:
- document loading
- embeddings
- vector database
- RAG pipeline
- Streamlit deployment
- LangSmith tracing

> After installation completes, restart the kernel/runtime and run all cells from the top.

In [44]:
print("Installing required packages...\n")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    streamlit \
    langsmith \
    python-dotenv \
    tiktoken

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

Installing required packages...


Installation complete.
Please restart the kernel/runtime before running the next cell.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\nanda\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Cell 2 — Configuration

This is the main configuration cell for the notebook.

Here you can:
- choose your LLM provider
- select the model you want to use
- update related settings if needed

All remaining cells will automatically use this configuration.

In [45]:
LLM_PROVIDER = "groq"
LLM_MODEL = "llama-3.3-70b-versatile"

CORPUS_PATH = r"C:\Users\nanda\Desktop\rag_projects\zyro-dynamics-hr-corpus"

print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {LLM_MODEL}")
print(f"Corpus: {CORPUS_PATH}")

Provider: groq
Model: llama-3.3-70b-versatile
Corpus: C:\Users\nanda\Desktop\rag_projects\zyro-dynamics-hr-corpus


## Cell 3 — Imports

This cell imports all required libraries for:
- document loading
- text chunking
- embeddings
- vector search
- prompt handling
- LangSmith tracing

> Run this cell without modifying anything.

In [46]:
import os, json, time, csv
from cryptography.fernet import Fernet
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import traceable

print("Imports loaded successfully.")

Imports loaded successfully.


## Cell 4 — API Keys + LangSmith Setup

This cell loads:
- your LLM API key
- LangSmith API key
- environment configuration

LangSmith tracing is enabled automatically for monitoring and debugging your RAG pipeline.

> Add the required API keys before running this cell.
> This section is pre-filled — no modifications needed.

In [47]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()

    if LLM_PROVIDER == "groq":
        os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
    elif LLM_PROVIDER == "gemini":
        os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    elif LLM_PROVIDER == "openai":
        os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")

    os.environ["LANGCHAIN_API_KEY"]    = secrets.get_secret("LANGCHAIN_API_KEY")
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"
    print("Running on Kaggle — secrets loaded!")

except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "#your project Name"

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

print("Environment configured successfully.")

Environment configured successfully.


## Cell 5 — Load Documents

### Your Task

Load all policy documents from the provided corpus directory using a PDF loader.

Store the loaded documents and print the total number loaded.

In [48]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader(CORPUS_PATH)
documents = loader.load()

print("Loaded documents:", len(documents))

Loaded documents: 39


In [49]:
print(CORPUS_PATH)
print(len(documents))

C:\Users\nanda\Desktop\rag_projects\zyro-dynamics-hr-corpus
39


## Cell 6 — Chunk Documents

### Your Task

Split the loaded documents into smaller chunks using `RecursiveCharacterTextSplitter`.

Store the generated chunks and print the total number created.

In [50]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")
print(len(chunks))
print(chunks[:2])

Created 112 chunks
112
[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T11:14:29+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-20T11:14:29+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\nanda\\Desktop\\rag_projects\\zyro-dynamics-hr-corpus\\00_Company_Profile.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Zyro Dynamics Pvt. Ltd.\nCompany Profile\nDoc Code: ZDL-CORP-001\nConfidential — For Internal Use Only\nPage 1\n Zyro Dynamics Pvt. Ltd.\n \n Navigate the Future\n Company Profile\n Document Code\n ZDL-CORP-001\n Version\n V.01\n Effective Date\n 01 April 2025\n Document Owner\n Corporate Communications\nCOMPANY OVERVIEW\nZyro Dynamics Pvt. Ltd. is a product-based enterprise software company headquartered in the United States. \nFounded in 2014 by Lamine Yamal and Aitana Bonmati, the company was built on a\n

## Cell 7 — Embeddings

### Your Task

Initialize an embedding model to convert document chunks into vector representations.

Use `HuggingFaceEmbeddings` for the implementation below.

In [51]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model initialized.")
print(type(embeddings))
test_embedding = embeddings.embed_query("hello world")
print(len(test_embedding))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5020.90it/s]


Embedding model initialized.
<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
384


In [52]:
print("Documents:", len(documents))
print("Chunks:", len(chunks))

if len(chunks) > 0:
    print("First chunk:")
    print(chunks[0])

print("Embeddings object:")
print(embeddings)
print(type(embeddings))

Documents: 39
Chunks: 112
First chunk:
page_content='Zyro Dynamics Pvt. Ltd.
Company Profile
Doc Code: ZDL-CORP-001
Confidential — For Internal Use Only
Page 1
 Zyro Dynamics Pvt. Ltd.
 
 Navigate the Future
 Company Profile
 Document Code
 ZDL-CORP-001
 Version
 V.01
 Effective Date
 01 April 2025
 Document Owner
 Corporate Communications
COMPANY OVERVIEW
Zyro Dynamics Pvt. Ltd. is a product-based enterprise software company headquartered in the United States. 
Founded in 2014 by Lamine Yamal and Aitana Bonmati, the company was built on a
single conviction: that organisations of all sizes deserved intelligent, intuitive software that could grow with them.
What started as a 12-person startup building HR automation tools has grown into a 4,500-strong company with a
presence across India, the Middle East, and North America.' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T11:14:29+00:00', 'author': '(anonymous)', 'keyw

## Cell 8 — Vector Store + Retriever

### Your Task

Build a FAISS vector store using the generated chunks and embeddings.

Then create a retriever for retrieving relevant document chunks during question answering.

In [53]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 12,
        "lambda_mult": 0.5,
    }
)

print("Vector store initialized.")
print("Documents:", len(documents))
print("Chunks:", len(chunks))

print("First chunk:")
print(chunks[0] if len(chunks) > 0 else "NO CHUNKS")

print("Embeddings:")
print(type(embeddings))

Vector store initialized.
Documents: 39
Chunks: 112
First chunk:
page_content='Zyro Dynamics Pvt. Ltd.
Company Profile
Doc Code: ZDL-CORP-001
Confidential — For Internal Use Only
Page 1
 Zyro Dynamics Pvt. Ltd.
 
 Navigate the Future
 Company Profile
 Document Code
 ZDL-CORP-001
 Version
 V.01
 Effective Date
 01 April 2025
 Document Owner
 Corporate Communications
COMPANY OVERVIEW
Zyro Dynamics Pvt. Ltd. is a product-based enterprise software company headquartered in the United States. 
Founded in 2014 by Lamine Yamal and Aitana Bonmati, the company was built on a
single conviction: that organisations of all sizes deserved intelligent, intuitive software that could grow with them.
What started as a 12-person startup building HR automation tools has grown into a 4,500-strong company with a
presence across India, the Middle East, and North America.' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T11:14:29+00:00', 'aut

In [54]:
import traceback

try:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    print("SUCCESS")
except Exception as e:
    print("ERROR TYPE:", type(e))
    print("ERROR:", e)
    traceback.print_exc()

SUCCESS


## Cell 9 — LLM Initialization

The language model is initialized automatically using the configuration from Cell 2.

You may optionally adjust parameters such as:
- `temperature`
- `max_tokens`

based on your preferred response style and performance.

> This cell is pre-filled — modify only if needed.

In [55]:
if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_output_tokens=512
    )

elif LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

else:
    raise ValueError("Unsupported LLM provider.")

print("LLM initialized.")

LLM initialized.


In [56]:
import os

os.environ["GROQ_API_KEY"] = "YOUR_NEW_GROQ_KEY"
os.environ["LANGCHAIN_API_KEY"] = "YOUR_NEW_LANGSMITH_KEY"

print("GROQ:", bool(os.getenv("GROQ_API_KEY")))
print("LANGCHAIN:", bool(os.getenv("LANGCHAIN_API_KEY")))

GROQ: True
LANGCHAIN: True


## Cell 10 — Build the RAG Chain

### Your Task

Implement the RAG pipeline using LCEL and enable LangSmith tracing.

In [57]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are Zyro Dynamics HR Assistant.

Answer ONLY from the provided context.

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

@traceable
def rag_chain(question: str):
    docs = retriever.invoke(question)

    context = format_docs(docs)

    chain = (
        RAG_PROMPT
        | llm
        | StrOutputParser()
    )

    answer = chain.invoke({
        "context": context,
        "question": question
    })

    return answer

## Cell 11 — Guardrails

### Your Task

Implement handling for out-of-scope questions before routing queries through the RAG pipeline.

In [58]:
REFUSAL_MESSAGE = (
    "I can only answer HR-related questions "
    "using Zyro Dynamics policy documents."
)

def ask_bot(question: str):
    docs = retriever.invoke(question)

    if len(docs) == 0:
        return {"answer": REFUSAL_MESSAGE}

    answer = rag_chain(question)

    return {"answer": answer}

## Cell 12 — Test the Bot

### Your Task

Test your RAG pipeline using a few sample questions of your choice.

In [59]:
test_questions = [
    "How many earned leave days are allowed?",
    "Can employees work from home?",
    "What is the probation period?"
]

for i, q in enumerate(test_questions, 1):
    print(f"Q{i}: {q}")

    result = ask_bot(q)

    print(result["answer"])
    print("-" * 60)

Q1: How many earned leave days are allowed?


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Cell 13 — LangSmith Trace URL

Generate and copy your shareable LangSmith trace URL for submission.

> This cell is pre-filled — just run it and follow the instructions.

In [ ]:
print("""
HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!
""")

## Cell 14 — Streamlit App

### Your Task

Build and deploy a Streamlit chatbot application for your RAG pipeline.

Save your implementation as `app.py`.

In [ ]:
app_code = """
# TODO: Build your Streamlit chatbot application

import streamlit as st

# your code here
"""

with open("app.py", "w") as f:
    f.write(app_code.strip())

print("app.py created.")

## Cell 15 — Evaluation

Evaluation inputs are loaded automatically at runtime.

> Do not modify this cell.

In [ ]:
_Q = [
    ("Q01", "gAAAAABqE-m-EnBhR94RLAsyCD5YUOimCgpyxnGmrg3N29dvcCChh_LbQzGhacqtB6Rg9ySTN-aO4eS5nnSSqgvslxWg3T2XNxvKRw9BoZOGB8sSrPpeXOqPKhdprAkvepa0Ef13rK84Lx_QKNPq5AMeO2zweDFo-UGpOZ1yFV_k0NbpkP0MshR9BpjCI4QpkDSx9QH95aeCK8sqSIkcM8wOFRs1hRD_tV-Jg4XmeHLm4jW6wpCWQRBF-XWIHTwCE3Tod-Zfj-nIFpPe3sNmXFDNY_L5g8aAiw=="),
    ("Q02", "gAAAAABqE-m-iGIUkxaPu-TWqkoQqfrY1QvCn-VC445z8EzeRjBVVSjcBgTYC-OS2QVoM37Oh8tFkJdLJcdivCIg9-jTJ72Vy24BQwagKYrIJlkNBr9yectRVtDZ_X24PWpsbIdMcelH1a6VBz9XXmJ19-0HvqFT0kTeEQEyjzKL2BmtoSHOquqe74xGFhpWD-fI1Cshfxk9EXwgA4poqi7JJ3ovja5pVM18uwfNAmcNacnQRtFTAm6x1JmXKSYVeBSbgpOv1zjEEC-0XfVhF0Wtwli0hRZHhA=="),
    ("Q03", "gAAAAABqE-m-qhjI3OCH68smnD4afuA_GmeOO8rI6R79iaPeodfwbt4NTlWhlbSfgr8BP9ZNAi5yczk65fgsIgbRXQ9AkAVDE2NOD11Aqt6U_NqURkjBQpzn3gzTQNj2qNwtkhx71-l8uYIfZLu8Z-Nv4aAkEaFTKCDp4DWgCaFJbe90TCA2fGUVnDiaI1_0ID87AHR-yYRwTaKYiWI7PiCQWFVm22NGx3cwX_uvMouAEXLX2sw_o3s="),
    ("Q04", "gAAAAABqE-m-qVKLekYizIYVBejJAmZYhT0zftdQzC0nbFt6BAJM52tiRsM0y5pcEfTl7y2bKwjFBSBwj3ik1P1yPTz6mP2h1xHEWoeJxPHdvujlZXJv8ObZO0PbHSPMk6xtnEmEqPAfPLzxjOzu63P3K_0eFdpgR48fUbcQwZt7yZkGzOPqYuUDAE7CBmvgvwRfwymkEzTD8ESt0vYvZdmoYjV7sbScmhoxYbWmjMatFvOzha6D1YA="),
    ("Q05", "gAAAAABqE-m-KRbrY2MpEseeszU46iQWHzbzwOO5-t10vHJrdQOKeaVwPxyp9kiBDCS1Fa5MJyQoTOp2pdEtw9LtUbCEJ_56caOBjtBgngLz4kvcodhVECBLBuD6vsCaQlopu0SardsvA3slA379M8nrcyuuea3dJ97FPlOdQs2b70BRPyOkyNH0nKGqBwQzBlAW7B-ucZwf9dDPPAw-xUTfR3ekIqXReQ=="),
    ("Q06", "gAAAAABqE-m-EYfgWBpxkb_5hGOvvBsAdBu5367Nd5d4uT_6EEAaTeCidG99u5XJ5vcZatZpoj5RjmfrY5O1XNObuApuq_ZFah_StEcLHB31Ow6WRrZpikDGUFJkC-ZfY0TggJzDFvdtwQsIttqNW5js0LMS-74V-AUx0UCi4bABm1vOMGBKP2qGyGTfyh2wfETTw4nNhbac"),
    ("Q07", "gAAAAABqE-m-cZLyG6To-HyWWdEYu42VgbV9c_SCWXt4qJE02YrOFvfMntuBTf-CVXt3MhJWFzrukGMR0-Brla1QMVbefRelzpJqkY2TsIQ3Tcc5MZ0BH6ornHjZAnOd9Iozf1f755EC8hBase1XtbhThrKgYJRKWPxaxKd-nkLK3XuabtmEF8r0bZtTyKVjYNBUWPT--lKJb-pXvw3p3zJ0z6utBLWicmBhgdJvGMoOQCsCLrxi6jrtHZzka7Me7Vm6UUhwSkdz"),
    ("Q08", "gAAAAABqE-m-sxXijCcjguEWTh7qgKt7BX4cbUfFdUwAz6VqSoU4fTnYXUhf-dVQdCKa1lhgc7ZZatU5Pu9iuQHG-ApZCOw2yR-PkZnuY9L7uR02CCJoWYhFQelqYEWYA5uONridoCzD8kh2yqwUSVInEFfBuB2cYgyPobRnP_yRvtaFtLakrMy0fsCZH_zfyrOMVkdF5GoHdPu67XzoEj806x4aS8DJ4ysYFuwNb9zkhhceq_CsU08="),
    ("Q09", "gAAAAABqE-m-nDGYgCF3fSWs2tM39pdnsBua61Ht1ruTZ_NOUmju6AxbGU6WB8HzLEHKQkkCnxc4ka2DohiUSLwVDrWG2ZnGggyt7OnI6D43ovjDBsMhW2jQPaz9zaHua25abfEqF4V1ZioQrdL7lz3D0qzDsjXl4Kw5RY2g3kaDakb62Cb6Dt8badoS-t4Bd_fEAp49t09FH_qwLp_ZTotiFsKFy6QADA=="),
    ("Q10", "gAAAAABqE-m-PwoVsLjWO4nbO8W_65P-UNNF7SjdNZL4sRN-G72eHygPuGyggXwVG8G7HJ2ZmrtCYuNg-rtWH_iuyexPQLVG0EqKr0ZQswJox4iauvFf014qlqr5vC_TtdwHGcMiZsyWZpJauDTffKDm_QJHrGElPUUunCFgX8356s1yMocleGXUBfcZ8B73A5LIALAXRIBpKyt707qYlLhwOG1vhsdR74q21NS0-n0skLZIy7z0pLM="),
    ("Q11", "gAAAAABqE-m-1BAGkhsZEDnkbSwAAwusmnMKdn2gvIM5tltaZ1W-eoKtvbPNu8rkAlOOiOW-9_NobJqDFKDO3J7zCPwWuEdGxwgYpX5sxh2Rg4ngR5R5WDnQsQTPIRHXJkkaN1ufNhvbQ-XOn2Z1QPci8118ByVpkAR5kZTUXOFIZ1IgHP2hbvO4E81GB9CTs9HiZvHAsAnS"),
    ("Q12", "gAAAAABqE-m-NrwI-KspXny9JlQqBEW_eB9jE6bGmnin6IX6SdcB9ol1gR7CmzczDKE6A7XHDOJW20tVHAlGFw-q-J6cWrTajK_mJTv00aHllSozrKiThojuxxnSjhgOhgtNKU5mh7zoz2d2uLo7p-Kl32m4IU6PRsm0kZceID-ZH5ZRw7w4h1qSZOufZO2HvKkR9LtfCQXk"),
    ("Q13", "gAAAAABqE-m-Xr56G8qaFfk3BIVQeDzP5mpahd7wZQ5vGR11AN_sxU1ZzjoPfbSdLmrrhFHEI8S8KhXfjOWZQoMJToWSsnhjZQdrRj0wujH38p2VOZLqqZYSmOflVEQm29z9pAXx_iltLWZLNGf8QsMtZWuo-3SsWt6R2mGvOMBTDj5hCzaq842_r1eupRQJJ1dnTSmNPskW"),
    ("Q14", "gAAAAABqE-m--oxJAL26EQ6bMS5vmgI0pWMWjgbG49qNZu8K_pIiDrp3ro1YFlVvBXOOJ6bSpI7lxz-OXmNrVFkSfJlVf4PchVKfWdddKVT85AMxUHo3PYD15IGV476RznHCiD59twp7x_E6HOF7AFUGiWcsO9Ph63Tfcvh3aJzF7Hk_NPEHcIaaEU9ki2eccYXehJJ3tkmr"),
    ("Q15", "gAAAAABqE-m-3JNAfb2dmCF-2XlNe-F1AaeXybgSJ4DwHtn9o52TEryPYgu-6m70Ivn7izeLy4h44AVbHL_3cv-MWfAwFYp7ct3lvF7dL1QbmhntyeY4c7l0CVPsc-mv8WuY04tpB2XPtHE_0ytl9tQlqAGonC2esnpMbSzgvZPdSw9eHnm5k2Jkh0FbgjLKNWxjdX3Uv2aYDiqOeLMQKZsMMteZzJcwHQ=="),
]

eval_questions = [
    {"question_id": qid, "question": fernet.decrypt(enc.encode()).decode()}
    for qid, enc in _Q
]

print(f"{len(eval_questions)} evaluation questions loaded.")

## Cell 16 — Generate `submission.csv`

Generate your final `submission.csv` file for submission.

> Do not modify this cell.

In [ ]:
import re

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("=" * 50)
print("Submission Generator")
print("=" * 50)

streamlit_link = input("Streamlit App URL: ").strip()
langsmith_link = input("LangSmith Trace URL: ").strip()

link_errors = []

if not STREAMLIT_PATTERN.match(streamlit_link):
    link_errors.append("Invalid Streamlit URL.")

if not LANGSMITH_PATTERN.match(langsmith_link):
    link_errors.append("Invalid LangSmith URL.")

if link_errors:
    print("\n".join(link_errors))
    raise ValueError("Please correct the links and re-run the cell.")

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

## Cell 17 — Final Checklist

Verify your submission files and links before submitting on Kaggle.

> This cell is pre-filled — just run it.

In [ ]:
import re, csv, os

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):

    with open("submission.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    count = len(rows)

    has_fields = all(
        all(
            k in r
            for k in [
                "question_id",
                "question_enc",
                "answer_enc",
                "streamlit_link",
                "langsmith_link"
            ]
        )
        for r in rows
    )

    sl_valid = all(
        STREAMLIT_PATTERN.match(r["streamlit_link"].strip())
        for r in rows
    )

    ll_valid = all(
        LANGSMITH_PATTERN.match(r["langsmith_link"].strip())
        for r in rows
    )

    print(f"submission.csv found ({count} rows)")
    print(f"Required columns present: {has_fields}")
    print(f"Streamlit links valid: {sl_valid}")
    print(f"LangSmith links valid: {ll_valid}")

    if not sl_valid or not ll_valid:
        print("\nPlease regenerate submission.csv with valid links.")

else:
    print("submission.csv not found. Run the previous cell first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")